# SCEPTRE on-target CRISPRi knockdown across D0–D3 — corrected SCEPTRE environment

This notebook reproduces the on-target SCEPTRE analysis design in `ipsc_ec_sceptre.ipynb` and makes one overlaid plotnine figure for **NF2, LATS2, AMOTL2 and TAOK1** across **D0, D1, D2 and D3**.

## Environment design

Use the following two Sherlock environments:

- **Notebook kernel and Python/plotnine:** `/oak/stanford/groups/engreitz/Users/tri/envs/sc-dl`
- **R and SCEPTRE:** `/oak/stanford/groups/engreitz/Users/tri/envs/sceptre_env/bin/Rscript`

The notebook runs with the `sc-dl` Python kernel and launches the SCEPTRE stage through the **direct `Rscript` executable** above. It no longer invokes `conda run`, and it does not use the `rpy2` environment.

The installed SCEPTRE in `sceptre_env` is version 0.10.2, whereas official `fold_change` and `se_fold_change` output begins in version 0.10.3. To avoid modifying or destabilizing the existing environment, the notebook installs the exact **SCEPTRE v0.10.3 tag** into this private R library when needed:

```text
/oak/stanford/groups/engreitz/Users/tri/Rlibs/sceptre_0.10.3
```

The subprocess receives that directory through `R_LIBS_USER`, so SCEPTRE v0.10.3 is loaded ahead of the environment's v0.10.2 copy. The install is skipped on subsequent runs after the exact version is verified. Set `INSTALL_PRIVATE_SCEPTRE_IF_NEEDED=False` only after v0.10.3 is already accessible to this Rscript.

## Figure encoding

- All four time points are displayed on **one axis**, horizontally offset within each target gene.
- **Shape** identifies time point: D0 circle, D1 square, D2 triangle and D3 diamond.
- **Open symbols at 1** show the control reference.
- **Filled symbols** show the SCEPTRE fold-change estimate for cells assigned a targeting guide.
- **Orange** indicates within-day BH-adjusted SCEPTRE `q ≤ 0.05`; black indicates `q > 0.05`.
- Grey bars are approximate 95% model-based intervals derived from SCEPTRE v0.10.3 `fold_change` and `se_fold_change` outputs.

## Analysis retained from the original notebook

For each day separately, the notebook uses the original raw response-count and gRNA-count transfer files, imports the data as high MOI, tests the on-target target–response pairs, uses two-sided tests and complement controls, assigns gRNAs with the mixture method, applies the original mitochondrial QC threshold, runs the calibration check, and then runs the discovery analysis. It writes new result files and never overwrites the historical `TargetKD.csv` files.

The SCEPTRE p-values remain conditional-resampling-based. The displayed intervals are model-based standard-error intervals and are not obtained by inverting the resampling test; the two uncertainty summaries need not agree exactly.

In [ ]:
import os
from pathlib import Path

class cfg:
    TARGETS = ["NF2", "LATS2", "AMOTL2", "TAOK1"]
    DAYS = ["d0", "d1", "d2", "d3"]

    # Correct Sherlock environments.
    SCEPTRE_ENV = Path("/oak/stanford/groups/engreitz/Users/tri/envs/sceptre_env")
    SCEPTRE_RSCRIPT = SCEPTRE_ENV / "bin/Rscript"
    SC_DL_ENV = Path("/oak/stanford/groups/engreitz/Users/tri/envs/sc-dl")
    REQUIRE_SC_DL_KERNEL = True

    # Use an exact private SCEPTRE 0.10.3 installation without replacing 0.10.2
    # inside sceptre_env. The first run installs it when absent; later runs verify/reuse it.
    SCEPTRE_PRIVATE_R_LIB = Path(
        "/oak/stanford/groups/engreitz/Users/tri/Rlibs/sceptre_0.10.3"
    )
    SCEPTRE_REQUIRED_VERSION = "0.10.3"
    SCEPTRE_GITHUB_REPO = "Katsevich-Lab/sceptre"
    SCEPTRE_GITHUB_REF = "v0.10.3"
    INSTALL_PRIVATE_SCEPTRE_IF_NEEDED = True
    CRAN_REPOSITORY = "https://cloud.r-project.org"

    # Paths copied from the original SCEPTRE notebook.
    NORMALIZED_DATA_DIR = Path(
        "/oak/stanford/groups/engreitz/Users/opushkar/cc_perturb/"
        "2.preprocessing_tonys_pipeline/2.normalized_data/normalized_data"
    )
    BASEDIR = Path(
        "/oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks"
    )
    GUIDE_METADATA_TSV = BASEDIR / "300genes_guide_metadata_v43.tsv"
    OLD_COMBINED_CSV = BASEDIR / "combined_TargetKD.csv"
    OUTDIR = BASEDIR / "sceptre_on_target_KD_D0_D3_overlay_plotnine_sceptre_env_v2"

    # Input preparation and SCEPTRE execution.
    FORCE_REBUILD_TRANSFER = False
    FORCE_RERUN_SCEPTRE = False
    RUN_CALIBRATION_CHECK = True
    SAVE_SCEPTRE_RDS = False
    R_PARALLEL = False  # Matches the original notebook and avoids cluster oversubscription.
    RANDOM_SEED = 20260922
    RESULT_FILENAME = "TargetKD_v0.10.3_with_SE.csv"

    # Statistical display.
    Q_THRESHOLD = 0.05
    CI_LEVEL = 0.95
    DAY_OFFSETS = {"d0": -0.27, "d1": -0.09, "d2": 0.09, "d3": 0.27}
    DAY_LABELS = {"d0": "D0", "d1": "D1", "d2": "D2", "d3": "D3"}
    DAY_SHAPES = {"D0": "o", "D1": "s", "D2": "^", "D3": "D"}

    # Plot settings.
    FIGSIZE = (8.7, 3.6)
    EXPORT_DPI = 600
    FONT_SIZE = 9
    POINT_SIZE = 2.8
    POINT_STROKE = 0.8
    CI_LINEWIDTH = 0.65
    CI_CAP_WIDTH = 0.07
    SIGNIFICANT_COLOR = "#D95F02"
    NS_COLOR = "#222222"
    CONTROL_EDGE = "#777777"
    CI_COLOR = "#777777"
    REFERENCE_LINE_COLOR = "#B0B0B0"

    TITLE = "On-target CRISPRi knockdown across D0–D3"
    Y_LABEL = "Target-gene expression relative to control"
    X_LABEL = "Target gene"
    PANEL_LETTER = ""

for day in cfg.DAYS:
    if day not in cfg.DAY_OFFSETS or day not in cfg.DAY_LABELS:
        raise ValueError(f"Missing plotting configuration for {day}.")
if len(set(cfg.TARGETS)) != len(cfg.TARGETS):
    raise ValueError("TARGETS must be unique.")

## 1. Validate the Python kernel and direct SCEPTRE Rscript

Run this notebook with the **`sc-dl` Python kernel**. The plotting and transfer stages use Python packages from:

```text
/oak/stanford/groups/engreitz/Users/tri/envs/sc-dl
```

The SCEPTRE stage is a separate subprocess launched directly through:

```text
/oak/stanford/groups/engreitz/Users/tri/envs/sceptre_env/bin/Rscript
```

The notebook does not use `conda run`, does not use `$OAK/Users/tri/envs/rpy2`, and does not require R cell magic.

If `sc-dl` is not registered as a Jupyter kernel, run this once in a Sherlock terminal:

```bash
/oak/stanford/groups/engreitz/Users/tri/envs/sc-dl/bin/python -m ipykernel install --user \
  --name sc-dl --display-name "Python (sc-dl)"
```

On first execution, the notebook checks the SCEPTRE version visible to the direct Rscript. If exact v0.10.3 is not available, it installs the official `v0.10.3` GitHub tag into the private library configured above. This does **not** replace the v0.10.2 package inside `sceptre_env`. A new R process is used for the analysis, so no Jupyter-kernel restart is required after installation.

In [ ]:
import csv
import gc
import hashlib
import importlib.metadata
import json
import os
import re
import subprocess
import sys
import textwrap
import warnings
import zipfile
from datetime import datetime, timezone

import matplotlib as mpl
from matplotlib import font_manager
import muon as mu
import numpy as np
import pandas as pd
import scipy
import scipy.io as sio
from scipy import sparse
from scipy.stats import norm
from IPython.display import display, Markdown

try:
    import plotnine as p9
    from plotnine import (
        ggplot, aes, geom_point, geom_errorbar, geom_hline,
        scale_fill_manual, scale_color_manual, scale_shape_manual,
        scale_x_continuous, scale_y_continuous, coord_cartesian,
        labs, theme_classic, theme, element_text, element_blank,
        element_line, guides, guide_legend,
    )
except ImportError as exc:
    raise ImportError(
        "plotnine is required in the sc-dl kernel. Select the kernel backed by "
        f"{cfg.SC_DL_ENV / 'bin/python'} and restart the notebook."
    ) from exc

OUTDIR = Path(cfg.OUTDIR)
OUTDIR.mkdir(parents=True, exist_ok=True)


def _resolved(pathlike):
    return Path(pathlike).expanduser().resolve()


current_prefix = _resolved(sys.prefix)
expected_prefix = _resolved(cfg.SC_DL_ENV)
if cfg.REQUIRE_SC_DL_KERNEL and current_prefix != expected_prefix:
    raise RuntimeError(
        "This notebook must run with the sc-dl Python kernel.\n"
        f"Current sys.prefix: {current_prefix}\n"
        f"Required environment: {expected_prefix}\n"
        "Select/register the kernel using the command in the preceding markdown cell."
    )
if not cfg.SCEPTRE_ENV.is_dir():
    raise FileNotFoundError(f"SCEPTRE environment not found: {cfg.SCEPTRE_ENV}")
if not cfg.SCEPTRE_RSCRIPT.is_file():
    raise FileNotFoundError(f"Direct SCEPTRE Rscript not found: {cfg.SCEPTRE_RSCRIPT}")
if not cfg.SC_DL_ENV.is_dir():
    raise FileNotFoundError(f"sc-dl environment not found: {cfg.SC_DL_ENV}")

available_fonts = {font.name for font in font_manager.fontManager.ttflist}
AI_FONT = next(
    (name for name in ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"]
     if name in available_fonts),
    "DejaVu Sans",
)
mpl.rcParams.update({
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "text.usetex": False,
    "font.family": "sans-serif",
    "font.sans-serif": [AI_FONT],
})

print("Output directory:", OUTDIR)
print("Notebook Python:", sys.executable)
print("Notebook environment:", current_prefix)
print("SCEPTRE environment:", cfg.SCEPTRE_ENV)
print("Direct Rscript:", cfg.SCEPTRE_RSCRIPT)
print("Private R library:", cfg.SCEPTRE_PRIVATE_R_LIB)
print("Required SCEPTRE version:", cfg.SCEPTRE_REQUIRED_VERSION)
print("Font:", AI_FONT)
print("plotnine:", p9.__version__)
print("Python:", sys.version.split()[0])

## 2. Validate or recreate the four SCEPTRE transfer folders

The original notebook transferred each day's MuData into Matrix Market files plus names, covariates and mappings. This cell reuses those files when complete. If a folder is missing or incomplete, it recreates it from the original normalized H5MU file while preserving the original definitions:

- response matrix: `GEX.layers['raw_counts']`, transposed to genes × cells;
- gRNA matrix: `guide.X`, transposed to guides × cells;
- batch: `GEX.obs['sample_id']`;
- response/gRNA depth and complexity fields copied exactly as in the original transfer table;
- mitochondrial percentage: `GEX.obs['pct_counts_mt']`;
- gRNA target: `intended_target_name` from `300genes_guide_metadata_v43.tsv`;
- response mapping: `GEX.var['symbol']` paired with the Ensembl response ID.

No cells are silently removed during transfer, matching the original notebook.


In [ ]:
TRANSFER_FILENAMES = [
    "response_matrix.mtx",
    "grna_matrix.mtx",
    "gene_names.csv",
    "guide_names.csv",
    "cell_names.csv",
    "covariates.csv",
    "guide_mapping.csv",
    "gene_symbol_map.csv",
]


def transfer_dir(day: str) -> Path:
    return cfg.BASEDIR / f"{day}_sceptre_transfer"


def transfer_paths(day: str) -> dict[str, Path]:
    directory = transfer_dir(day)
    return {name: directory / name for name in TRANSFER_FILENAMES}


def complete_transfer(day: str) -> bool:
    return all(path.is_file() and path.stat().st_size > 0 for path in transfer_paths(day).values())


def prepare_transfer(day: str, guide_metadata: pd.DataFrame | None = None) -> dict[str, Path]:
    paths = transfer_paths(day)
    directory = transfer_dir(day)
    directory.mkdir(parents=True, exist_ok=True)

    if complete_transfer(day) and not cfg.FORCE_REBUILD_TRANSFER:
        print(f"{day.upper()}: reusing complete transfer folder: {directory}")
        return paths

    if guide_metadata is None:
        if not cfg.GUIDE_METADATA_TSV.is_file():
            raise FileNotFoundError(cfg.GUIDE_METADATA_TSV)
        guide_metadata = pd.read_csv(cfg.GUIDE_METADATA_TSV, sep="\t")

    required_meta = {"guide_id", "intended_target_name"}
    missing_meta = required_meta - set(guide_metadata.columns)
    if missing_meta:
        raise ValueError(f"Guide metadata lacks columns: {sorted(missing_meta)}")
    if guide_metadata["guide_id"].duplicated().any():
        duplicated = guide_metadata.loc[guide_metadata["guide_id"].duplicated(), "guide_id"].head().tolist()
        raise ValueError(f"Duplicated guide_id values in metadata, e.g. {duplicated}")

    h5mu_path = cfg.NORMALIZED_DATA_DIR / f"{day}_normalized_gex_and_guide.h5mu"
    if not h5mu_path.is_file():
        raise FileNotFoundError(h5mu_path)

    print(f"{day.upper()}: rebuilding transfer files from {h5mu_path}")
    mdata = mu.read(h5mu_path)
    if "GEX" not in mdata.mod or "guide" not in mdata.mod:
        raise KeyError(f"{h5mu_path} must contain GEX and guide modalities.")
    gex = mdata["GEX"]
    guide = mdata["guide"]

    if not gex.obs_names.equals(guide.obs_names):
        raise ValueError(f"GEX and guide cell orders differ for {day}; do not guess an alignment.")
    if "raw_counts" not in gex.layers:
        raise KeyError(f"GEX.layers['raw_counts'] is missing for {day}.")
    required_obs = {
        "sample_id", "guides_per_cell", "guide_umi_counts",
        "n_genes_by_counts", "total_counts", "pct_counts_mt",
    }
    missing_obs = required_obs - set(gex.obs.columns)
    if missing_obs:
        raise KeyError(f"Missing GEX.obs fields for {day}: {sorted(missing_obs)}")
    if "symbol" not in gex.var.columns:
        raise KeyError(f"GEX.var['symbol'] is missing for {day}.")

    sio.mmwrite(paths["response_matrix.mtx"], sparse.csr_matrix(gex.layers["raw_counts"]).T)
    sio.mmwrite(paths["grna_matrix.mtx"], sparse.csr_matrix(guide.X).T)
    pd.Series(gex.var_names.astype(str)).to_csv(paths["gene_names.csv"], index=False, header=False)
    pd.Series(guide.var_names.astype(str)).to_csv(paths["guide_names.csv"], index=False, header=False)
    pd.Series(gex.obs_names.astype(str)).to_csv(paths["cell_names.csv"], index=False, header=False)

    covariates = pd.DataFrame({
        "batch": gex.obs["sample_id"].to_numpy(),
        "grna_n_nonzero": gex.obs["guides_per_cell"].to_numpy(),
        "grna_n_umis": gex.obs["guide_umi_counts"].to_numpy(),
        "response_n_nonzero": gex.obs["n_genes_by_counts"].to_numpy(),
        "response_n_umis": gex.obs["total_counts"].to_numpy(),
        "response_p_mito": gex.obs["pct_counts_mt"].to_numpy(),
    }, index=gex.obs_names.astype(str))
    covariates.to_csv(paths["covariates.csv"])

    lookup = guide_metadata.set_index("guide_id")
    guide_ids = pd.Index(guide.var_names.astype(str), name="grna_id")
    missing_guides = guide_ids.difference(lookup.index.astype(str))
    if len(missing_guides):
        raise ValueError(f"{len(missing_guides)} guides are absent from metadata; examples: {missing_guides[:5].tolist()}")
    # Reindex after converting the lookup index to string to avoid mixed-type surprises.
    lookup = lookup.copy()
    lookup.index = lookup.index.astype(str)
    mapped_targets = lookup.loc[guide_ids, "intended_target_name"]
    if mapped_targets.isna().any():
        bad = guide_ids[mapped_targets.isna().to_numpy()][:5].tolist()
        raise ValueError(f"Unmapped guide targets remain for {day}; examples: {bad}")
    guide_mapping = pd.DataFrame({
        "grna_id": guide_ids,
        "grna_target": mapped_targets.astype(str).to_numpy(),
    })
    guide_mapping.to_csv(paths["guide_mapping.csv"], index=False)

    if gex.var["symbol"].isna().any():
        raise ValueError(f"Missing gene symbols remain in {day}; do not create ambiguous on-target pairs.")
    gene_map = pd.DataFrame({
        "response_id": gex.var_names.astype(str),
        "grna_target": gex.var["symbol"].astype(str).to_numpy(),
    })
    gene_map.to_csv(paths["gene_symbol_map.csv"], index=False)

    missing_targets = sorted(set(cfg.TARGETS) - set(gene_map["grna_target"]))
    if missing_targets:
        raise ValueError(f"Target response genes absent from {day}: {missing_targets}")

    del mdata, gex, guide
    gc.collect()
    if not complete_transfer(day):
        missing = [str(path) for path in paths.values() if not path.is_file() or path.stat().st_size == 0]
        raise IOError(f"Transfer creation failed for {day}: {missing}")
    print(f"{day.upper()}: transfer files written and verified.")
    return paths


guide_metadata = None
if cfg.FORCE_REBUILD_TRANSFER or not all(complete_transfer(day) for day in cfg.DAYS):
    if not cfg.GUIDE_METADATA_TSV.is_file():
        raise FileNotFoundError(cfg.GUIDE_METADATA_TSV)
    guide_metadata = pd.read_csv(cfg.GUIDE_METADATA_TSV, sep="\t")

TRANSFER_PATHS = {day: prepare_transfer(day, guide_metadata) for day in cfg.DAYS}
print("All transfer folders are complete.")

## 3. Install/verify SCEPTRE v0.10.3, then run or reuse each day's discovery analysis

The direct R executable is:

```text
/oak/stanford/groups/engreitz/Users/tri/envs/sceptre_env/bin/Rscript
```

The next cell first runs a small installer/preflight script. It checks `.libPaths()`, verifies the package version and package location, and—when needed—installs the exact official `v0.10.3` tag into:

```text
/oak/stanford/groups/engreitz/Users/tri/Rlibs/sceptre_0.10.3
```

The existing v0.10.2 installation in `sceptre_env` is not removed or overwritten. The analysis subprocess receives the private directory through `R_LIBS_USER` and verifies both `packageVersion("sceptre") == "0.10.3"` and the resolved package path before loading the analysis inputs.

The SCEPTRE analysis retains the original choices:

- high-MOI import;
- on-target response–target pairs only;
- two-sided testing;
- union integration of guides targeting the same gene;
- complement controls;
- conditional randomization testing with skew-normal approximation;
- the original/default high-MOI adjustment formula;
- mixture-model gRNA assignment;
- `p_mito_threshold = 0.25` in QC;
- calibration check before discovery analysis;
- BH correction within each day's complete on-target discovery family.

The official fold-change standard errors are available from v0.10.3. The notebook deliberately does not manufacture a standard error from the SCEPTRE p-value or from `z_orig`. Existing complete v0.10.3 result files are reused unless `FORCE_RERUN_SCEPTRE=True`.

In [ ]:
def run_streamed(command: list[str], *, env: dict[str, str], log_path: Path) -> None:
    """Run a subprocess, stream combined stdout/stderr, and save the complete log."""
    print("Running external command:")
    print(" ".join(command))
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            command,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_handle.write(line)
            log_handle.flush()
        return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


def direct_rscript_command(script_path: Path) -> list[str]:
    """Use the user's exact sceptre_env Rscript; never route through conda run."""
    script_path = Path(script_path).expanduser().resolve()
    if not cfg.SCEPTRE_RSCRIPT.is_file():
        raise FileNotFoundError(cfg.SCEPTRE_RSCRIPT)
    return [str(cfg.SCEPTRE_RSCRIPT), "--vanilla", str(script_path)]


# Keep numerical libraries single-threaded inside the R subprocess. SCEPTRE itself
# is also run with parallel=False, matching the original notebook.
R_SUBPROCESS_ENV = os.environ.copy()
R_SUBPROCESS_ENV.update({
    "R_LIBS_USER": str(cfg.SCEPTRE_PRIVATE_R_LIB),
    "SCEPTRE_PRIVATE_R_LIB": str(cfg.SCEPTRE_PRIVATE_R_LIB),
    "SCEPTRE_REQUIRED_VERSION": cfg.SCEPTRE_REQUIRED_VERSION,
    "SCEPTRE_GITHUB_REPO": cfg.SCEPTRE_GITHUB_REPO,
    "SCEPTRE_GITHUB_REF": cfg.SCEPTRE_GITHUB_REF,
    "SCEPTRE_INSTALL_IF_NEEDED": str(cfg.INSTALL_PRIVATE_SCEPTRE_IF_NEEDED).lower(),
    "SCEPTRE_CRAN_REPOSITORY": cfg.CRAN_REPOSITORY,
    "OMP_NUM_THREADS": "1",
    "OPENBLAS_NUM_THREADS": "1",
    "MKL_NUM_THREADS": "1",
    "VECLIB_MAXIMUM_THREADS": "1",
    "NUMEXPR_NUM_THREADS": "1",
})

INSTALL_SCRIPT_PATH = OUTDIR / "ensure_sceptre_0.10.3_private.R"
INSTALL_LOG_PATH = OUTDIR / "ensure_sceptre_0.10.3_private.log"

INSTALL_SCRIPT_TEXT = r"""
parse_bool <- function(x) tolower(trimws(x)) %in% c("true", "t", "1", "yes", "y")
env_or_default <- function(name, default) {
  value <- Sys.getenv(name)
  if (nzchar(value)) value else default
}
private_lib <- env_or_default(
  "SCEPTRE_PRIVATE_R_LIB",
  "/oak/stanford/groups/engreitz/Users/tri/Rlibs/sceptre_0.10.3"
)
required_version_text <- env_or_default("SCEPTRE_REQUIRED_VERSION", "0.10.3")
required_version <- package_version(required_version_text)
repo <- env_or_default("SCEPTRE_GITHUB_REPO", "Katsevich-Lab/sceptre")
ref <- env_or_default("SCEPTRE_GITHUB_REF", "v0.10.3")
install_if_needed <- parse_bool(env_or_default("SCEPTRE_INSTALL_IF_NEEDED", "true"))
cran_repository <- env_or_default("SCEPTRE_CRAN_REPOSITORY", "https://cloud.r-project.org")
output_dir <- env_or_default("SCEPTRE_OUTPUT_DIR", private_lib)

if (!nzchar(private_lib)) stop("SCEPTRE_PRIVATE_R_LIB is empty.")
dir.create(private_lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(unique(c(private_lib, .libPaths())))

package_location <- function(package_name) {
  tryCatch(find.package(package_name, quiet = TRUE), error = function(e) "")
}
package_version_from_path <- function(path) {
  if (!nzchar(path)) return(package_version("0.0.0"))
  description_path <- file.path(path, "DESCRIPTION")
  if (!file.exists(description_path)) return(package_version("0.0.0"))
  package_version(read.dcf(description_path, fields = "Version")[1, 1])
}

current_path <- package_location("sceptre")
current_version <- package_version_from_path(current_path)
cat("R executable:", file.path(R.home("bin"), "R"), "\n")
cat("Initial .libPaths():", paste(.libPaths(), collapse = " | "), "\n")
cat("Initially resolved SCEPTRE path:", ifelse(nzchar(current_path), current_path, "<not found>"), "\n")
cat("Initially resolved SCEPTRE version:", as.character(current_version), "\n")

need_install <- current_version != required_version
if (need_install) {
  if (!install_if_needed) {
    stop(
      paste0(
        "Exact SCEPTRE ", required_version_text, " is not available. ",
        "Set INSTALL_PRIVATE_SCEPTRE_IF_NEEDED=True or install ", repo, "@", ref,
        " into ", private_lib, "."
      )
    )
  }
  cat("Installing exact SCEPTRE tag", ref, "into", private_lib, "\n")
  remotes_path <- package_location("remotes")
  if (!nzchar(remotes_path)) {
    install.packages(
      "remotes", repos = cran_repository, lib = private_lib,
      dependencies = c("Depends", "Imports", "LinkingTo")
    )
  }
  remotes::install_github(
    repo = repo,
    ref = ref,
    lib = private_lib,
    dependencies = FALSE,
    upgrade = "never",
    force = TRUE,
    build_vignettes = FALSE,
    quiet = FALSE
  )
}

# Re-resolve after installation. This script never attaches the old namespace.
.libPaths(unique(c(private_lib, .libPaths())))
final_path <- package_location("sceptre")
final_version <- package_version_from_path(final_path)
if (!nzchar(final_path)) stop("SCEPTRE is still unavailable after preflight/install.")
if (final_version != required_version) {
  stop(
    paste0(
      "Expected exact SCEPTRE ", required_version_text,
      " but resolved ", final_version, " at ", final_path, "."
    )
  )
}
private_norm <- normalizePath(private_lib, winslash = "/", mustWork = TRUE)
final_norm <- normalizePath(final_path, winslash = "/", mustWork = TRUE)
if (!startsWith(final_norm, paste0(private_norm, "/"))) {
  warning(
    paste0(
      "Exact SCEPTRE version was found outside the private library at ", final_norm,
      ". The version is acceptable, but the package location differs from the preferred private library."
    )
  )
}
cat("Verified SCEPTRE version:", as.character(final_version), "\n")
cat("Verified SCEPTRE package path:", final_path, "\n")
writeLines(
  c(
    paste0("sceptre_version=", as.character(final_version)),
    paste0("sceptre_path=", final_path),
    paste0("private_library=", private_lib),
    paste0("r_version=", R.version.string)
  ),
  file.path(output_dir, "sceptre_private_library_preflight.txt")
)
"""

R_SUBPROCESS_ENV["SCEPTRE_OUTPUT_DIR"] = str(OUTDIR)
INSTALL_SCRIPT_PATH.write_text(INSTALL_SCRIPT_TEXT, encoding="utf-8")
INSTALL_COMMAND_USED = direct_rscript_command(INSTALL_SCRIPT_PATH)
run_streamed(INSTALL_COMMAND_USED, env=R_SUBPROCESS_ENV, log_path=INSTALL_LOG_PATH)

preflight_record = OUTDIR / "sceptre_private_library_preflight.txt"
if not preflight_record.is_file() or preflight_record.stat().st_size == 0:
    raise IOError("SCEPTRE private-library preflight did not produce its verification record.")
print(preflight_record.read_text(encoding="utf-8"))

# Variables consumed by the main analysis R script.
R_SUBPROCESS_ENV.update({
    "SCEPTRE_BASEDIR": str(cfg.BASEDIR),
    "SCEPTRE_DAYS": ",".join(cfg.DAYS),
    "SCEPTRE_RESULT_FILENAME": cfg.RESULT_FILENAME,
    "SCEPTRE_FORCE_RERUN": str(cfg.FORCE_RERUN_SCEPTRE).lower(),
    "SCEPTRE_RUN_CALIBRATION": str(cfg.RUN_CALIBRATION_CHECK).lower(),
    "SCEPTRE_SAVE_RDS": str(cfg.SAVE_SCEPTRE_RDS).lower(),
    "SCEPTRE_PARALLEL": str(cfg.R_PARALLEL).lower(),
    "SCEPTRE_Q_THRESHOLD": repr(cfg.Q_THRESHOLD),
    "SCEPTRE_RANDOM_SEED": str(cfg.RANDOM_SEED),
})

R_SCRIPT_PATH = OUTDIR / "run_sceptre_with_official_se.R"
R_LOG_PATH = OUTDIR / "run_sceptre_with_official_se.log"

In [ ]:
R_SCRIPT_TEXT = r'''
private_lib <- Sys.getenv("SCEPTRE_PRIVATE_R_LIB")
if (!nzchar(private_lib)) stop("SCEPTRE_PRIVATE_R_LIB is empty.")
.libPaths(unique(c(private_lib, .libPaths())))

suppressPackageStartupMessages({
  library(sceptre)
  library(Matrix)
})

parse_bool <- function(x) tolower(trimws(x)) %in% c("true", "t", "1", "yes", "y")
base_dir <- Sys.getenv("SCEPTRE_BASEDIR")
days <- strsplit(Sys.getenv("SCEPTRE_DAYS"), ",", fixed = TRUE)[[1]]
result_filename <- Sys.getenv("SCEPTRE_RESULT_FILENAME")
force_rerun <- parse_bool(Sys.getenv("SCEPTRE_FORCE_RERUN"))
run_calibration <- parse_bool(Sys.getenv("SCEPTRE_RUN_CALIBRATION"))
save_rds <- parse_bool(Sys.getenv("SCEPTRE_SAVE_RDS"))
run_parallel <- parse_bool(Sys.getenv("SCEPTRE_PARALLEL"))
q_threshold <- as.numeric(Sys.getenv("SCEPTRE_Q_THRESHOLD"))
base_seed <- as.integer(Sys.getenv("SCEPTRE_RANDOM_SEED"))
required_version <- package_version(Sys.getenv("SCEPTRE_REQUIRED_VERSION"))
installed_version <- packageVersion("sceptre")
installed_path <- find.package("sceptre")

cat("SCEPTRE version:", as.character(installed_version), "\n")
cat("SCEPTRE package path:", installed_path, "\n")
cat("R library paths:", paste(.libPaths(), collapse = " | "), "\n")
if (installed_version != required_version) {
  stop(paste0("Expected exact SCEPTRE ", required_version, " but loaded ", installed_version, " from ", installed_path, "."))
}
if (!is.finite(q_threshold) || q_threshold <= 0 || q_threshold >= 1) {
  stop("SCEPTRE_Q_THRESHOLD must lie strictly between 0 and 1.")
}

required_result_columns <- c(
  "response_id", "grna_target", "n_nonzero_trt", "n_nonzero_cntrl",
  "pass_qc", "p_value", "fold_change", "se_fold_change",
  "log_2_fold_change", "significant"
)

to_logical_strict <- function(x, column_name) {
  if (is.logical(x)) return(x)
  values <- tolower(trimws(as.character(x)))
  out <- rep(NA, length(values))
  out[values %in% c("true", "t", "1")] <- TRUE
  out[values %in% c("false", "f", "0")] <- FALSE
  unknown <- is.na(out) & !is.na(x)
  if (any(unknown)) {
    stop(paste("Cannot parse", column_name, "as logical; examples:", paste(head(unique(values[unknown])), collapse = ", ")))
  }
  out
}

normalise_result <- function(result, day_label) {
  result <- as.data.frame(result, stringsAsFactors = FALSE)
  missing <- setdiff(required_result_columns, names(result))
  if (length(missing) > 0) {
    stop(paste(day_label, "result lacks required columns:", paste(missing, collapse = ", ")))
  }
  numeric_columns <- c(
    "n_nonzero_trt", "n_nonzero_cntrl", "p_value",
    "fold_change", "se_fold_change", "log_2_fold_change"
  )
  for (column in numeric_columns) result[[column]] <- as.numeric(result[[column]])
  result$pass_qc <- to_logical_strict(result$pass_qc, "pass_qc")
  result$significant <- to_logical_strict(result$significant, "significant")
  result$q_bh <- NA_real_
  eligible <- which(result$pass_qc & is.finite(result$p_value))
  if (length(eligible) == 0) stop(paste(day_label, "has no QC-passing finite p-values."))
  result$q_bh[eligible] <- p.adjust(result$p_value[eligible], method = "BH")
  result$q_significant <- !is.na(result$q_bh) & result$q_bh <= q_threshold
  result$day <- day_label

  finite_fc <- !is.na(result$pass_qc) & result$pass_qc &
    is.finite(result$fold_change) & result$fold_change > 0 &
    is.finite(result$log_2_fold_change)
  if (any(finite_fc, na.rm = TRUE)) {
    discrepancy <- max(abs(log2(result$fold_change[finite_fc]) - result$log_2_fold_change[finite_fc]))
    if (!is.finite(discrepancy) || discrepancy > 1e-6) {
      stop(paste(day_label, "fold_change and log_2_fold_change are inconsistent; max discrepancy =", discrepancy))
    }
  }
  result
}

read_names <- function(path) {
  as.character(read.csv(path, header = FALSE, check.names = FALSE, stringsAsFactors = FALSE)[, 1])
}

for (day_index in seq_along(days)) {
  day <- days[[day_index]]
  day_dir <- file.path(base_dir, paste0(day, "_sceptre_transfer"))
  updated_path <- file.path(day_dir, result_filename)
  historical_path <- file.path(day_dir, "TargetKD.csv")
  result <- NULL
  result_source <- NULL

  if (!force_rerun) {
    for (candidate in c(updated_path, historical_path)) {
      if (!file.exists(candidate)) next
      candidate_result <- tryCatch(
        read.csv(candidate, check.names = FALSE, stringsAsFactors = FALSE),
        error = function(e) NULL
      )
      if (!is.null(candidate_result) && all(required_result_columns %in% names(candidate_result))) {
        result <- normalise_result(candidate_result, day)
        result_source <- candidate
        break
      }
    }
  }

  if (is.null(result)) {
    cat("\n", day, ": running SCEPTRE from transfer inputs.\n", sep = "")
    required_inputs <- c(
      "response_matrix.mtx", "grna_matrix.mtx", "gene_names.csv",
      "guide_names.csv", "cell_names.csv", "covariates.csv",
      "guide_mapping.csv", "gene_symbol_map.csv"
    )
    missing_inputs <- required_inputs[!file.exists(file.path(day_dir, required_inputs))]
    if (length(missing_inputs) > 0) {
      stop(paste(day, "is missing transfer files:", paste(missing_inputs, collapse = ", ")))
    }

    response_mat <- as(readMM(file.path(day_dir, "response_matrix.mtx")), "dgCMatrix")
    grna_mat <- as(readMM(file.path(day_dir, "grna_matrix.mtx")), "dgCMatrix")
    gene_names <- read_names(file.path(day_dir, "gene_names.csv"))
    guide_names <- read_names(file.path(day_dir, "guide_names.csv"))
    cell_names <- read_names(file.path(day_dir, "cell_names.csv"))
    if (length(gene_names) != nrow(response_mat)) stop(paste(day, "gene-name count mismatch."))
    if (length(guide_names) != nrow(grna_mat)) stop(paste(day, "guide-name count mismatch."))
    if (length(cell_names) != ncol(response_mat) || length(cell_names) != ncol(grna_mat)) {
      stop(paste(day, "cell-name count mismatch."))
    }
    rownames(response_mat) <- gene_names
    colnames(response_mat) <- cell_names
    rownames(grna_mat) <- guide_names
    colnames(grna_mat) <- cell_names

    covariates <- read.csv(
      file.path(day_dir, "covariates.csv"), row.names = 1,
      check.names = FALSE, stringsAsFactors = FALSE
    )
    if (!all(cell_names %in% rownames(covariates))) stop(paste(day, "covariates omit cells."))
    covariates <- covariates[cell_names, , drop = FALSE]
    if ("response_p_mito" %in% names(covariates)) {
      names(covariates)[names(covariates) == "response_p_mito"] <- "mt_percent"
    }
    reserved_columns <- c(
      "response_n_nonzero", "response_n_umis", "grna_n_nonzero", "grna_n_umis"
    )
    covariates <- covariates[, !(names(covariates) %in% reserved_columns), drop = FALSE]

    guide_mapping <- read.csv(
      file.path(day_dir, "guide_mapping.csv"),
      check.names = FALSE, stringsAsFactors = FALSE
    )
    gene_map <- read.csv(
      file.path(day_dir, "gene_symbol_map.csv"),
      check.names = FALSE, stringsAsFactors = FALSE
    )

    set.seed(base_seed + day_index - 1L)
    sceptre_object <- import_data(
      response_matrix = response_mat,
      grna_matrix = grna_mat,
      grna_target_data_frame = guide_mapping,
      extra_covariates = covariates,
      moi = "high"
    )

    discovery_pairs <- merge(guide_mapping, gene_map, by = "grna_target")
    discovery_pairs <- unique(discovery_pairs[, c("grna_target", "response_id")])
    if (nrow(discovery_pairs) == 0) stop(paste(day, "has no on-target discovery pairs."))

    analysis_formula <- formula(
      ~ log(response_n_nonzero) + log(response_n_umis) +
        log(grna_n_nonzero) + log(grna_n_umis) + batch
    )
    sceptre_object <- set_analysis_parameters(
      sceptre_object = sceptre_object,
      discovery_pairs = discovery_pairs,
      side = "both",
      grna_integration_strategy = "union",
      formula_object = analysis_formula,
      resampling_approximation = "skew_normal",
      control_group = "complement",
      resampling_mechanism = "crt",
      multiple_testing_method = "BH",
      multiple_testing_alpha = q_threshold
    )
    sceptre_object <- assign_grnas(
      sceptre_object = sceptre_object,
      method = "mixture",
      parallel = run_parallel
    )
    sceptre_object <- run_qc(
      sceptre_object = sceptre_object,
      p_mito_threshold = 0.25
    )

    if (run_calibration) {
      sceptre_object <- run_calibration_check(
        sceptre_object = sceptre_object,
        parallel = run_parallel
      )
      calibration_result <- get_result(
        sceptre_object = sceptre_object,
        analysis = "run_calibration_check"
      )
      write.csv(
        as.data.frame(calibration_result),
        file.path(day_dir, "calibration_result_updated_sceptre.csv"),
        row.names = FALSE
      )
    }

    sceptre_object <- run_discovery_analysis(
      sceptre_object = sceptre_object,
      output_amount = 1,
      parallel = run_parallel
    )
    result <- get_result(
      sceptre_object = sceptre_object,
      analysis = "run_discovery_analysis"
    )
    result <- normalise_result(result, day)
    result_source <- "fresh_sceptre_rerun"

    if (save_rds) {
      saveRDS(sceptre_object, file.path(day_dir, "sceptre_object_v0.10.3_with_SE.rds"))
    }
    writeLines(
      capture.output(sessionInfo()),
      file.path(day_dir, "sceptre_sessionInfo_v0.10.3.txt")
    )
    writeLines(
      capture.output(print(sceptre_object)),
      file.path(day_dir, "sceptre_analysis_summary_v0.10.3.txt")
    )
    rm(
      response_mat, grna_mat, covariates, guide_mapping, gene_map,
      discovery_pairs, sceptre_object
    )
    gc()
  } else {
    cat(day, ": reusing result with standard errors from ", result_source, "\n", sep = "")
  }

  write.csv(result, updated_path, row.names = FALSE)
  writeLines(
    c(
      paste0("sceptre_version=", as.character(installed_version)),
      paste0("sceptre_path=", installed_path),
      paste0("result_source=", result_source),
      paste0("q_threshold=", q_threshold),
      paste0("n_pairs=", nrow(result)),
      paste0("n_qc_passing=", sum(result$pass_qc, na.rm = TRUE))
    ),
    file.path(day_dir, "TargetKD_with_SE_v0.10.3_metadata.txt")
  )
  cat(day, ": wrote ", updated_path, "\n", sep = "")
}
'''

R_SCRIPT_PATH.write_text(R_SCRIPT_TEXT, encoding="utf-8")
if not R_SCRIPT_PATH.is_file() or R_SCRIPT_PATH.stat().st_size == 0:
    raise IOError(f"Failed to write R script: {R_SCRIPT_PATH}")

R_COMMAND_USED = direct_rscript_command(R_SCRIPT_PATH)
run_streamed(R_COMMAND_USED, env=R_SUBPROCESS_ENV, log_path=R_LOG_PATH)

if not R_LOG_PATH.is_file() or R_LOG_PATH.stat().st_size == 0:
    raise IOError(f"SCEPTRE subprocess did not produce a log: {R_LOG_PATH}")
print("SCEPTRE subprocess completed successfully.")
print("R script:", R_SCRIPT_PATH)
print("R log:", R_LOG_PATH)


## 4. Construct SCEPTRE-aligned estimates and official-SE confidence intervals

For each day and target gene, let

\[
\widehat F
\]

be SCEPTRE v0.10.3's `fold_change` estimate, and let

\[
s_F
\]

be its official `se_fold_change` output. The displayed control reference is fixed at 1, and the filled point is \(\widehat F\).

To keep the interval positive, the notebook forms an approximate Wald interval on the log2 scale. By the delta method,

\[
\widehat\theta = \log_2(\widehat F),
\qquad
s_\theta \approx \frac{s_F}{\widehat F\ln 2}.
\]

For confidence level \(1-\alpha\), the plotted relative-expression interval is

\[
\left[
2^{\widehat\theta-z_{1-\alpha/2}s_\theta},
\;2^{\widehat\theta+z_{1-\alpha/2}s_\theta}
\right].
\]

This is a model-based interval for the fold-change estimator. The orange/black significance call is based separately on the conditional-resampling SCEPTRE p-value followed by BH correction across **all QC-passing on-target pairs within the same day**, before selecting the four displayed genes. The notebook does not infer an SE from the p-value or from `z_orig`.

In [ ]:


def parse_bool_series(series: pd.Series, name: str) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    text = series.astype(str).str.strip().str.lower()
    mapping = {"true": True, "t": True, "1": True, "false": False, "f": False, "0": False}
    parsed = text.map(mapping)
    if parsed.isna().any():
        examples = series[parsed.isna()].astype(str).drop_duplicates().head().tolist()
        raise ValueError(f"Cannot parse {name} as Boolean; examples: {examples}")
    return parsed.astype(bool)

def bh_adjust(p_values: np.ndarray) -> np.ndarray:
    """Benjamini-Hochberg adjusted p values, preserving original order."""
    p = np.asarray(p_values, dtype=float)
    if p.ndim != 1 or len(p) == 0 or not np.isfinite(p).all() or np.any((p < 0) | (p > 1)):
        raise ValueError("BH input must be a nonempty finite vector in [0, 1].")
    order = np.argsort(p, kind="mergesort")
    ranked = p[order]
    n = len(p)
    adjusted_ranked = ranked * n / np.arange(1, n + 1)
    adjusted_ranked = np.minimum.accumulate(adjusted_ranked[::-1])[::-1]
    adjusted_ranked = np.clip(adjusted_ranked, 0, 1)
    adjusted = np.empty(n, dtype=float)
    adjusted[order] = adjusted_ranked
    return adjusted


required_columns = {
    "response_id", "grna_target", "n_nonzero_trt", "n_nonzero_cntrl",
    "pass_qc", "p_value", "fold_change", "se_fold_change",
    "log_2_fold_change", "q_bh", "q_significant", "day",
}
all_results = []
q_audits = []
for day in cfg.DAYS:
    path = transfer_dir(day) / cfg.RESULT_FILENAME
    if not path.is_file():
        raise FileNotFoundError(path)
    result = pd.read_csv(path)
    missing = required_columns - set(result.columns)
    if missing:
        raise ValueError(f"{path} lacks columns: {sorted(missing)}")
    if set(result["day"].astype(str)) != {day}:
        raise ValueError(f"{path} has an unexpected day label.")
    for column in ["p_value", "fold_change", "se_fold_change", "log_2_fold_change", "q_bh"]:
        result[column] = pd.to_numeric(result[column], errors="coerce")
    result["pass_qc"] = parse_bool_series(result["pass_qc"], "pass_qc")
    result["q_significant"] = parse_bool_series(result["q_significant"], "q_significant")

    eligible = result["pass_qc"] & np.isfinite(result["p_value"])
    q_check = bh_adjust(result.loc[eligible, "p_value"].to_numpy(float))
    max_q_difference = float(np.max(np.abs(q_check - result.loc[eligible, "q_bh"].to_numpy(float))))
    if max_q_difference > 1e-12:
        raise ValueError(f"BH audit failed for {day}: max |q_python - q_R| = {max_q_difference}")
    q_audits.append({"day": day, "n_family": int(eligible.sum()), "max_q_difference": max_q_difference})
    all_results.append(result)

all_results = pd.concat(all_results, ignore_index=True)
pd.DataFrame(q_audits).to_csv(OUTDIR / "q_value_audit.csv", index=False)
all_results.to_csv(OUTDIR / "all_on_target_sceptre_results_v0.10.3_with_SE.csv", index=False)

selected = all_results[all_results["grna_target"].isin(cfg.TARGETS)].copy()
expected_keys = {(day, gene) for day in cfg.DAYS for gene in cfg.TARGETS}
actual_keys = set(zip(selected["day"], selected["grna_target"]))
if actual_keys != expected_keys or len(selected) != len(expected_keys):
    missing = sorted(expected_keys - actual_keys)
    duplicates = selected.duplicated(["day", "grna_target"], keep=False)
    raise ValueError(
        f"Expected exactly one row per day/gene. Missing={missing}; "
        f"duplicated rows={selected.loc[duplicates, ['day', 'grna_target']].to_dict('records')}"
    )
if not selected["pass_qc"].all():
    failed = selected.loc[~selected["pass_qc"], ["day", "grna_target"]].to_dict("records")
    raise ValueError(f"Displayed pairs failed SCEPTRE pairwise QC: {failed}")

numeric = selected[["fold_change", "se_fold_change", "log_2_fold_change", "q_bh"]].to_numpy(float)
if not np.isfinite(numeric).all():
    raise ValueError("Selected pairs have missing or nonfinite effect/SE/q values.")
if np.any(selected["fold_change"].to_numpy(float) <= 0):
    raise ValueError("SCEPTRE fold changes must be positive.")
if np.any(selected["se_fold_change"].to_numpy(float) < 0):
    raise ValueError("SCEPTRE fold-change standard errors must be nonnegative.")

selected["log2_fc_recomputed"] = np.log2(selected["fold_change"].to_numpy(float))
selected["log2_fc_discrepancy"] = (
    selected["log2_fc_recomputed"] - selected["log_2_fold_change"].to_numpy(float)
).abs()
if selected["log2_fc_discrepancy"].max() > 1e-6:
    raise ValueError("fold_change and log_2_fold_change are inconsistent for displayed pairs.")

z_value = float(norm.ppf(1 - (1 - cfg.CI_LEVEL) / 2))
selected["se_log2_fold_change_delta"] = (
    selected["se_fold_change"] /
    (selected["fold_change"] * np.log(2.0))
)
selected["ci_low"] = 2.0 ** (
    selected["log_2_fold_change"] - z_value * selected["se_log2_fold_change_delta"]
)
selected["ci_high"] = 2.0 ** (
    selected["log_2_fold_change"] + z_value * selected["se_log2_fold_change_delta"]
)
selected["ci_method"] = "SCEPTRE_v0.10.3_se_fold_change_delta_method_log2_Wald"
selected["q_significant_figure"] = selected["q_bh"] <= cfg.Q_THRESHOLD
selected["knockdown_pct"] = 100.0 * (1.0 - selected["fold_change"])
selected["day_label"] = selected["day"].map(cfg.DAY_LABELS)
selected["gene_order"] = selected["grna_target"].map({gene: i for i, gene in enumerate(cfg.TARGETS)})
selected["day_order"] = selected["day"].map({day: i for i, day in enumerate(cfg.DAYS)})
selected = selected.sort_values(["gene_order", "day_order"], kind="stable").reset_index(drop=True)
selected.to_csv(OUTDIR / "sceptre_4genes_effects_and_CIs.csv", index=False)

display_columns = [
    "day", "grna_target", "fold_change", "se_fold_change",
    "ci_low", "ci_high", "p_value", "q_bh", "knockdown_pct", "ci_method",
]
display(selected[display_columns])

## 5. Make one overlaid plotnine panel

Each gene occupies one x-axis group. The four days are horizontally offset within the group and distinguished by shape. For each day, the open symbol marks the control reference at 1 and the filled symbol marks the SCEPTRE fold-change estimate. Confidence bars are drawn only for the estimated fold change because the control value is the reference denominator, not a separately estimated SCEPTRE point in this display.


In [ ]:
plot_rows = []
gene_positions = {gene: index for index, gene in enumerate(cfg.TARGETS)}
for row in selected.itertuples(index=False):
    x_position = gene_positions[row.grna_target] + cfg.DAY_OFFSETS[row.day]
    common = {
        "target": row.grna_target,
        "day": row.day,
        "day_label": row.day_label,
        "x_plot": x_position,
        "p_value": row.p_value,
        "q_bh": row.q_bh,
        "q_significant": bool(row.q_significant_figure),
        "sceptre_fold_change": row.fold_change,
        "sceptre_se_fold_change": row.se_fold_change,
        "sceptre_log_2_fold_change": row.log_2_fold_change,
        "knockdown_pct": row.knockdown_pct,
        "ci_method": row.ci_method,
    }
    plot_rows.append({
        **common,
        "group": "control",
        "estimate": 1.0,
        "ci_low": np.nan,
        "ci_high": np.nan,
    })
    plot_rows.append({
        **common,
        "group": "targeting",
        "estimate": row.fold_change,
        "ci_low": row.ci_low,
        "ci_high": row.ci_high,
    })

plot_data = pd.DataFrame(plot_rows)
significant_label = f"q ≤ {cfg.Q_THRESHOLD:g}"
legend_order = ["control", "NS", significant_label]
plot_data["point_class"] = np.where(
    plot_data["group"].eq("control"),
    "control",
    np.where(plot_data["q_significant"], significant_label, "NS"),
)
plot_data["point_class"] = pd.Categorical(
    plot_data["point_class"], categories=legend_order, ordered=True
)
day_labels = [cfg.DAY_LABELS[day] for day in cfg.DAYS]
plot_data["day_label"] = pd.Categorical(
    plot_data["day_label"], categories=day_labels, ordered=True
)
plot_data = plot_data.sort_values(
    ["target", "day_label", "group"], kind="stable"
).reset_index(drop=True)
plot_data.to_csv(OUTDIR / "plot_source_data_control_and_targeting.csv", index=False)

targeting_data = plot_data[plot_data["group"].eq("targeting")].copy()
fill_values = {
    "control": "white",
    "NS": cfg.NS_COLOR,
    significant_label: cfg.SIGNIFICANT_COLOR,
}
edge_values = {
    "control": cfg.CONTROL_EDGE,
    "NS": cfg.NS_COLOR,
    significant_label: cfg.SIGNIFICANT_COLOR,
}

upper = float(max(1.0, targeting_data["ci_high"].max()))
y_max = max(1.08, upper * 1.08)
title = cfg.TITLE if not cfg.PANEL_LETTER else f"{cfg.PANEL_LETTER}  {cfg.TITLE}"

panel = (
    ggplot(plot_data, aes(x="x_plot", y="estimate"))
    + geom_hline(
        yintercept=1.0, linetype="dashed",
        color=cfg.REFERENCE_LINE_COLOR, size=0.5,
    )
    + geom_point(
        aes(shape="day_label", fill="point_class", color="point_class"),
        size=cfg.POINT_SIZE,
        stroke=cfg.POINT_STROKE,
        show_legend=True,
    )
    # Draw literal CI endpoints after the points so narrow intervals remain visible.
    + geom_errorbar(
        data=targeting_data,
        mapping=aes(x="x_plot", ymin="ci_low", ymax="ci_high"),
        inherit_aes=False,
        width=cfg.CI_CAP_WIDTH,
        color=cfg.CI_COLOR,
        size=cfg.CI_LINEWIDTH,
        show_legend=False,
    )
    + scale_shape_manual(values=cfg.DAY_SHAPES, name="Time point")
    + scale_fill_manual(
        values=fill_values,
        limits=legend_order,
        breaks=legend_order,
        drop=False,
        name="",
    )
    + scale_color_manual(
        values=edge_values,
        limits=legend_order,
        breaks=legend_order,
        drop=False,
        name="",
        guide=None,
    )
    + guides(
        fill=guide_legend(
            title="", order=1, nrow=1, byrow=True,
            override_aes={
                "shape": "o", "size": cfg.POINT_SIZE,
                "stroke": cfg.POINT_STROKE,
            },
        ),
        shape=guide_legend(
            title="Time point", order=2, nrow=1, byrow=True,
            override_aes={
                "fill": "white", "color": "#444444",
                "size": cfg.POINT_SIZE, "stroke": cfg.POINT_STROKE,
            },
        ),
    )
    + scale_x_continuous(
        breaks=list(range(len(cfg.TARGETS))),
        labels=cfg.TARGETS,
        expand=(0.08, 0.08),
    )
    + scale_y_continuous(expand=(0, 0))
    + coord_cartesian(
        xlim=(-0.55, len(cfg.TARGETS) - 0.45),
        ylim=(0, y_max),
        expand=False,
    )
    + labs(title=title, x=cfg.X_LABEL, y=cfg.Y_LABEL)
    + theme_classic(base_size=cfg.FONT_SIZE, base_family=AI_FONT)
    + theme(
        figure_size=cfg.FIGSIZE,
        dpi=120,
        svg_usefonts=True,
        panel_grid_major=element_blank(),
        panel_grid_minor=element_blank(),
        axis_text_x=element_text(rotation=35, ha="right", style="italic"),
        plot_title=element_text(weight="bold", ha="left"),
        legend_position="top",
        legend_direction="horizontal",
        legend_box="horizontal",
        legend_title=element_text(size=cfg.FONT_SIZE - 1),
        legend_text=element_text(size=cfg.FONT_SIZE - 1),
        axis_line=element_line(color="black", size=0.7),
        axis_ticks=element_line(color="black", size=0.5),
    )
)

figure_paths = []
for extension in ["pdf", "svg", "png"]:
    path = OUTDIR / f"sceptre_on_target_KD_D0_D3_overlay.{extension}"
    panel.save(
        filename=str(path),
        width=cfg.FIGSIZE[0],
        height=cfg.FIGSIZE[1],
        units="in",
        dpi=cfg.EXPORT_DPI,
        verbose=False,
    )
    if not path.is_file() or path.stat().st_size == 0:
        raise IOError(f"Figure export failed: {path}")
    figure_paths.append(path)

svg_text = (OUTDIR / "sceptre_on_target_KD_D0_D3_overlay.svg").read_text(
    encoding="utf-8", errors="ignore"
)
if "<text" not in svg_text:
    raise IOError("SVG export does not contain editable text elements.")

display(panel)
print("Saved figure files:")
for path in figure_paths:
    print(path)

## 6. Save figure legend, provenance and a recoverable output bundle


In [ ]:
control_description = "the control reference, fixed at a relative-expression value of 1"
legend_text = (
    "On-target CRISPRi knockdown of NF2, LATS2, AMOTL2 and TAOK1 across D0–D3. "
    "Time points are overlaid within each target-gene group and distinguished by symbol shape. "
    f"Open symbols show {control_description}; filled symbols show SCEPTRE v0.10.3 fold-change estimates for cells assigned guides targeting the indicated gene. "
    f"Grey bars show approximate {cfg.CI_LEVEL * 100:g}% Wald confidence intervals formed on the log2 fold-change scale by applying the delta method to SCEPTRE's official fold_change and se_fold_change outputs. "
    "These model-based intervals are not obtained by inverting SCEPTRE's conditional-resampling test. "
    f"Orange symbols indicate BH-adjusted SCEPTRE q ≤ {cfg.Q_THRESHOLD:g}; black symbols indicate q > {cfg.Q_THRESHOLD:g}. "
    "BH adjustment was performed separately within each time point across all QC-passing on-target discovery pairs before selecting the four displayed genes. "
    "The dashed horizontal line marks the control reference of 1."
)
(OUTDIR / "figure_legend.txt").write_text(legend_text + "\n", encoding="utf-8")


def file_record(path: Path) -> dict:
    stat = path.stat()
    return {
        "path": str(path),
        "size_bytes": stat.st_size,
        "mtime_ns": stat.st_mtime_ns,
    }

result_files = {day: transfer_dir(day) / cfg.RESULT_FILENAME for day in cfg.DAYS}
provenance = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "analysis_source": "ipsc_ec_sceptre.ipynb",
    "environment_strategy": "sc-dl Jupyter kernel plus direct sceptre_env/bin/Rscript subprocess",
    "sc_dl_env": str(cfg.SC_DL_ENV),
    "sc_dl_python": sys.executable,
    "sceptre_env": str(cfg.SCEPTRE_ENV),
    "sceptre_rscript": str(cfg.SCEPTRE_RSCRIPT),
    "sceptre_private_r_library": str(cfg.SCEPTRE_PRIVATE_R_LIB),
    "sceptre_required_version": cfg.SCEPTRE_REQUIRED_VERSION,
    "sceptre_github_ref": cfg.SCEPTRE_GITHUB_REF,
    "install_command": INSTALL_COMMAND_USED,
    "analysis_r_command": R_COMMAND_USED,
    "install_script": file_record(INSTALL_SCRIPT_PATH),
    "install_log": file_record(INSTALL_LOG_PATH),
    "sceptre_preflight": file_record(OUTDIR / "sceptre_private_library_preflight.txt"),
    "analysis_r_script": file_record(R_SCRIPT_PATH),
    "analysis_r_log": file_record(R_LOG_PATH),
    "sceptre_result_files": {day: file_record(path) for day, path in result_files.items()},
    "targets": cfg.TARGETS,
    "days": cfg.DAYS,
    "q_threshold": cfg.Q_THRESHOLD,
    "q_family": "all QC-passing on-target discovery pairs separately within each day",
    "ci_level": cfg.CI_LEVEL,
    "ci_method": (
        "delta-method Wald interval on log2(fold_change), using official SCEPTRE "
        "v0.10.3 se_log2 = se_fold_change / (fold_change * ln(2))"
    ),
    "control_reference": 1.0,
    "plotting_library": "plotnine",
    "plotnine_version": p9.__version__,
    "python_packages": {
        name: importlib.metadata.version(name)
        for name in ["numpy", "pandas", "scipy", "muon", "matplotlib", "plotnine", "mizani"]
    },
    "font": AI_FONT,
    "figure_files": [file_record(path) for path in figure_paths],
    "original_combined_result_present": cfg.OLD_COMBINED_CSV.is_file(),
}
(OUTDIR / "analysis_provenance.json").write_text(
    json.dumps(provenance, indent=2, default=str) + "\n",
    encoding="utf-8",
)

# Audit against the historical 0.10.2-derived combined table. Historical values
# are never substituted into the new plot.
if cfg.OLD_COMBINED_CSV.is_file():
    historical = pd.read_csv(cfg.OLD_COMBINED_CSV)
    if {"day", "grna_target", "log_2_fold_change"}.issubset(historical.columns):
        historical = historical[historical["grna_target"].isin(cfg.TARGETS)].copy()
        historical["day"] = historical["day"].map(
            lambda x: f"d{int(x)}" if str(x).replace(".0", "").isdigit() else str(x).lower()
        )
        historical = historical[["day", "grna_target", "log_2_fold_change"]].rename(
            columns={"log_2_fold_change": "historical_v0.10.2_log_2_fold_change"}
        )
        comparison = selected.merge(historical, on=["day", "grna_target"], how="left")
        comparison["v0.10.3_minus_historical_log2fc"] = (
            comparison["log_2_fold_change"]
            - comparison["historical_v0.10.2_log_2_fold_change"]
        )
        comparison.to_csv(OUTDIR / "historical_v0.10.2_vs_v0.10.3_effect_audit.csv", index=False)

manifest_rows = []
for path in sorted(OUTDIR.iterdir()):
    if not path.is_file() or path.suffix == ".zip" or path.name == "output_manifest.csv":
        continue
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    manifest_rows.append({"file": path.name, "bytes": path.stat().st_size, "sha256": digest})
pd.DataFrame(manifest_rows).to_csv(OUTDIR / "output_manifest.csv", index=False)

archive = OUTDIR / "sceptre_on_target_KD_D0_D3_overlay_outputs.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as package:
    for path in sorted(OUTDIR.iterdir()):
        if path.is_file() and path.suffix != ".zip":
            package.write(path, arcname=path.name)
with zipfile.ZipFile(archive) as package:
    bad_member = package.testzip()
    if bad_member is not None:
        raise IOError(f"ZIP integrity failure at {bad_member}")

print("Figure legend:\n", legend_text)
print("\nVerified output archive:", archive)

## Run instructions

1. Register/select the `sc-dl` Python kernel if needed:

```bash
/oak/stanford/groups/engreitz/Users/tri/envs/sc-dl/bin/python -m ipykernel install --user \
  --name sc-dl --display-name "Python (sc-dl)"
```

2. Open this notebook, select **Python (sc-dl)** and run from the top.
3. The notebook calls the exact executable:

```text
/oak/stanford/groups/engreitz/Users/tri/envs/sceptre_env/bin/Rscript
```

4. On the first run, it verifies or installs exact SCEPTRE v0.10.3 in:

```text
/oak/stanford/groups/engreitz/Users/tri/Rlibs/sceptre_0.10.3
```

This leaves the environment's v0.10.2 installation unchanged. If GitHub access is blocked on a compute node, run the preflight/install cell once on a Sherlock login node, then rerun the notebook. No Python-kernel restart is needed because SCEPTRE is launched in a new external R process.

5. Any day without a complete v0.10.3 `TargetKD_v0.10.3_with_SE.csv` is reanalyzed. Later runs reuse these files unless `FORCE_RERUN_SCEPTRE=True`.
6. The exact install script, install log, resolved package path/version, analysis R script, subprocess command and full R log are saved with the outputs.
7. The editable combined figure is written to:

```text
/oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks/
sceptre_on_target_KD_D0_D3_overlay_plotnine_sceptre_env_v2/
sceptre_on_target_KD_D0_D3_overlay.svg
```

The corresponding PDF, 600-dpi PNG, source-data tables, legend, provenance record and verified ZIP archive are saved in the same directory.